# `scheme_name` 02: noteworthy single-feature findings

**Purpose:** identify and discuss supported points that stand out after the
standard `high-cardinality-category` breakdown. Target relationships here are
exploratory and must be rechecked after the split is frozen.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_stage_directory():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "data" / "TrainingSetValues.csv").exists()
            and (candidate / "src" / "source_data_validation.py").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the stage-1-pump-it-up directory.")


stage_directory = find_stage_directory()
source_directory = str((stage_directory / "src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from predictor_audit import (
    analysis_categories,
    categorical_summary,
    categorical_target_profile,
    category_frequency_table,
    numeric_summary,
    numeric_target_summary,
    related_feature_summary,
    sentinel_mask,
    source_blank_mask,
    text_normalisation_summary,
)
from source_data_validation import (
    validate_aligned_ids,
    validate_label_frame,
    validate_raw_feature_schema,
)

data_directory = stage_directory / "data"
training_features = pd.read_csv(
    data_directory / "TrainingSetValues.csv",
    keep_default_na=False,
)
training_labels = pd.read_csv(
    data_directory / "TrainingSetLabels.csv",
    keep_default_na=False,
)
test_features = pd.read_csv(
    data_directory / "TestSetValues.csv",
    keep_default_na=False,
)

validate_raw_feature_schema(training_features)
validate_raw_feature_schema(test_features)
validate_label_frame(training_labels)
validate_aligned_ids(training_features, training_labels)

training_data = training_features.merge(
    training_labels,
    on="id",
    validate="one_to_one",
)

feature = 'scheme_name'
feature_metadata = {'order': 21, 'name': 'scheme_name', 'audit_type': 'high-cardinality-category', 'role': 'candidate', 'disposition': 'exclude raw one-hot value from the first baseline', 'finding': 'Nearly half the source values are blank and the recorded names are high-cardinality and inconsistent with scheme management.', 'decision': 'Preserve blank and literal sentinels separately; test only fold-fitted high-cardinality treatments.', 'risk': 'Scheme identity can memorise projects and geography.', 'sentinel_tokens': ['None', 'none', 'no scheme', 'not known'], 'related': [{'feature': 'scheme_management', 'reason': 'Scheme names map imperfectly to the management label.'}, {'feature': 'management', 'reason': 'Management gives a complete lower-cardinality alternative.'}, {'feature': 'funder', 'reason': 'Funders may repeatedly support named schemes.'}, {'feature': 'installer', 'reason': 'Installers may repeatedly construct named schemes.'}]}
feature_types = {'amount_tsh': 'numeric', 'date_recorded': 'date', 'funder': 'high-cardinality-category', 'gps_height': 'numeric', 'installer': 'high-cardinality-category', 'longitude': 'coordinate', 'latitude': 'coordinate', 'wpt_name': 'high-cardinality-category', 'num_private': 'numeric', 'basin': 'category', 'subvillage': 'high-cardinality-category', 'region': 'category', 'region_code': 'category', 'district_code': 'category', 'lga': 'category', 'ward': 'high-cardinality-category', 'population': 'numeric', 'public_meeting': 'binary', 'recorded_by': 'constant', 'scheme_management': 'category', 'scheme_name': 'high-cardinality-category', 'permit': 'binary', 'construction_year': 'year', 'extraction_type': 'category', 'extraction_type_group': 'category', 'extraction_type_class': 'category', 'management': 'category', 'management_group': 'category', 'payment': 'category', 'payment_type': 'category', 'water_quality': 'category', 'quality_group': 'category', 'quantity': 'category', 'quantity_group': 'category', 'source': 'category', 'source_type': 'category', 'source_class': 'category', 'waterpoint_type': 'category', 'waterpoint_type_group': 'category'}
assert feature in training_features.columns
print(
    f"Validated {len(training_features):,} training rows and "
    f"{len(test_features):,} test rows for {feature}."
)


Validated 59,400 training rows and 14,850 test rows for scheme_name.


## Supported target evidence


In [2]:
sentinel_tokens = ['None', 'none', 'no scheme', 'not known']
target_profile = categorical_target_profile(
    training_data,
    feature,
    minimum_support=100,
    sentinel_tokens=sentinel_tokens,
)
display(target_profile.head(20))

supported = target_profile.loc[target_profile["meets support threshold"]].copy()
non_functional_column = "non functional (%)"
if non_functional_column in supported:
    display(
        supported.sort_values(non_functional_column, ascending=False)
        .head(12)[["rows", non_functional_column]]
    )


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
scheme_name,,,,,
<missing/blank>,28166,True,51.44,7.11,41.45
k,685,True,55.04,16.79,28.18
none,669,True,63.68,4.48,31.84
borehole,546,True,37.36,4.76,57.88
chalinze wate,406,True,85.96,0.00,14.04
m,400,True,49.25,14.00,36.75
danida,379,True,52.51,4.49,43.01
government,320,True,46.88,10.00,43.12
bagamoyo wate,296,True,70.27,0.00,29.73


status_group,rows,non functional (%)
scheme_name,,
sinyanga water supplied sch,100,94.00
ngana water supplied scheme,270,72.59
maambreni gravity water supply,125,63.20
shallow well,119,60.50
borehole,546,57.88
s,154,50.00
government,320,43.12
danida,379,43.01
<missing/blank>,28166,41.45


## Observation

Nearly half the source values are blank and the recorded names are high-cardinality and inconsistent with scheme management.

## Interpretation

The supported single-feature patterns make this field worth the stated
treatment, but they do not prove causation or independent predictive value.
High-cardinality and geographic fields are especially vulnerable to
memorisation under a random split.

## Provisional decision

Preserve blank and literal sentinels separately; test only fold-fitted high-cardinality treatments.

**Risk to carry forward:** Scheme identity can memorise projects and geography.


In [3]:
decision_record = pd.DataFrame([{
    "feature": feature,
    "role": feature_metadata["role"],
    "disposition": feature_metadata["disposition"],
    "finding": feature_metadata["finding"],
    "decision": feature_metadata["decision"],
    "risk": feature_metadata["risk"],
}])
display(decision_record.set_index("feature"))


,role,disposition,finding,decision,risk
feature,,,,,
scheme_name,candidate,exclude raw one-hot value from the first baseline,Nearly half the source values are blank and th...,Preserve blank and literal sentinels separatel...,Scheme identity can memorise projects and geog...
